In [1]:
import numpy as np
import torch
import torch.nn.functional as F
from torch import Tensor
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset
from tqdm import tqdm

In [2]:
model_name = 'intfloat/multilingual-e5-large'
lang = "fr"
batch_size = 192

In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).cuda()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:86: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Load Collection Data

In [ ]:
collection_data = load_dataset("sschellhammer/CT26_Task1_SourceRetrievalForScientificWebClaims", "collection", token="<HF_TOKEN>")["collection"]
collection_keys = np.array(collection_data["pubkey"])

def create_text(example):
    text = f'passage: {example["title"]}. {example["venue"]}. {example["abstract"]}. {example["authors"]}'
    return {"text": text}

collection_data = collection_data.map(create_text)

def tokenize_with_padding(examples):
    return tokenizer(examples["text"], max_length=512, truncation=True, padding="max_length", return_tensors="pt")

collection_data = collection_data.map(tokenize_with_padding, batched=True, remove_columns=collection_data.column_names).with_format("torch", columns=["input_ids", "attention_mask"])

collection_data.json:   0%|          | 0.00/36.5M [00:00<?, ?B/s]

Generating collection split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

### Embed Collection Data

In [6]:
def average_pool(last_hidden_states: Tensor, attention_mask: Tensor) -> Tensor:
    last_hidden = last_hidden_states.masked_fill(~attention_mask[..., None].bool(), 0.0)
    return last_hidden.sum(dim=1) / attention_mask.sum(dim=1)[..., None]

embeddings = []
with torch.inference_mode():
    for batch in tqdm(DataLoader(collection_data, batch_size=batch_size, num_workers=10, prefetch_factor=2)):
        batch = {k: v.to(model.device) for k, v in batch.items()}
        outputs = model(**batch)
        batch_embeddings = average_pool(outputs.last_hidden_state, batch['attention_mask'])
        embeddings.append(batch_embeddings.detach().cpu())

collection_embeddings = torch.vstack(embeddings)
collection_embeddings = F.normalize(collection_embeddings, p=2, dim=1)

100%|██████████| 53/53 [03:04<00:00,  3.47s/it]


### Load Query Data

In [ ]:
data = load_dataset("sschellhammer/CT26_Task1_SourceRetrievalForScientificWebClaims", lang, token="<HF_TOKEN>")
train = data["train"]
train_labels = np.array(train["pubkey"])

dev = data["dev"]
dev_labels = np.array(dev["pubkey"])

fr_train.json: 0.00B [00:00, ?B/s]

fr_dev.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/2807 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/702 [00:00<?, ? examples/s]

In [8]:
def process_text(example):
    text = example["text"]
    text = f'query: {text}'
    return {"text": text}

train = train.map(process_text)
dev = dev.map(process_text)

def tokenize_with_padding(examples):
    return tokenizer(examples["text"], max_length=128, truncation=True, padding="max_length", return_tensors="pt")

train = train.map(tokenize_with_padding, batched=True, remove_columns=train.column_names).with_format("torch", columns=["input_ids", "attention_mask"])
dev = dev.map(tokenize_with_padding, batched=True, remove_columns=dev.column_names).with_format("torch", columns=["input_ids", "attention_mask"])

Map:   0%|          | 0/2807 [00:00<?, ? examples/s]

Map:   0%|          | 0/702 [00:00<?, ? examples/s]

Map:   0%|          | 0/2807 [00:00<?, ? examples/s]

Map:   0%|          | 0/702 [00:00<?, ? examples/s]

### Embed Query Data

In [9]:
train_embeddings = []
with torch.inference_mode():
    for batch in tqdm(DataLoader(train, batch_size=batch_size, num_workers=10, prefetch_factor=2)):
        batch = {k: v.to(model.device) for k, v in batch.items()}
        outputs = model(**batch)
        batch_embeddings = average_pool(outputs.last_hidden_state, batch['attention_mask'])
        train_embeddings.append(batch_embeddings.detach().cpu())

train_embeddings = torch.vstack(train_embeddings)
train_embeddings = F.normalize(train_embeddings, p=2, dim=1)

dev_embeddings = []
with torch.inference_mode():
    for batch in tqdm(DataLoader(dev, batch_size=batch_size, num_workers=10, prefetch_factor=2)):
        batch = {k: v.to(model.device) for k, v in batch.items()}
        outputs = model(**batch)
        batch_embeddings = average_pool(outputs.last_hidden_state, batch['attention_mask'])
        dev_embeddings.append(batch_embeddings.detach().cpu())

dev_embeddings = torch.vstack(dev_embeddings)
dev_embeddings = F.normalize(dev_embeddings, p=2, dim=1)

100%|██████████| 4/4 [00:03<00:00,  1.16it/s]


### Calculate MRR@5 Score

In [10]:
def mrr_at_5(cos_sims, labels):
    # Get top-5 predictions
    top5_idxs = np.argsort(-cos_sims, axis=1)[:, :5]
    top5_keys = collection_keys[top5_idxs]

    # Compare labels to top-5 predictions
    labels_2d = labels[:, None]
    matches_mask = (top5_keys == labels_2d)

    # Get position of correct predictions
    positions = np.argmax(matches_mask, axis=1)
    positions[~matches_mask.any(axis=1)] = -1
    positions = positions.astype(int)

    # Get reciprocal rank
    rr = np.zeros_like(positions, dtype=float)
    mask = positions >= 0
    rr[mask] = 1.0 / (positions[mask] + 1)

    return float(rr.mean())

In [11]:
train_cos_sims = (train_embeddings @ collection_embeddings.T).numpy()
dev_cos_sims = (dev_embeddings @ collection_embeddings.T).numpy()

print("MRR@5 Train:", mrr_at_5(train_cos_sims, train_labels))
print("MRR@5 Dev:", mrr_at_5(dev_cos_sims, dev_labels))

MRR@5 Train: 0.46831730198313737
MRR@5 Dev: 0.45584045584045585
